# Heights in metres

pyrosm reads OpenStreetMap levels, and each building counts its levels from its own ground floor. On a slope the levels of two buildings do not meet. In the centre of Helsinki, level 2 of the shopping centre Forum opens to the courtyard Kukontori, which is at street level on the uphill side of the block. Drawn by levels, that door hangs two floors above the street it opens to.

With a terrain model, pyrosm3d gives the walking network heights in metres: streets lie on the terrain, and floors are placed from the streets that they meet. This page reads the heights as data, draws the network in metres and draws a route through it.

A terrain model (a digital elevation model, DEM) is a raster of ground heights. pyrosm3d bundles and downloads none. `dem` takes a GeoTIFF or another raster file, a list of tiles, or a function `(longitudes, latitudes) -> heights` with the coordinates in EPSG:4326. Reading a file needs rasterio, which the `dem` extra installs:

```bash
pip install "pyrosm3d[dem] @ git+https://github.com/pyrosm/pyrosm3D@main"
```

The examples use the Helsinki files of [cafein.sampledata](https://pypi.org/project/cafein.sampledata/) (`pip install cafein.sampledata`): an OpenStreetMap extract of the capital region, and the elevation model of the National Land Survey of Finland in 10 m pixels. Each file is downloaded when it is first used.

## Read the network with heights

`osm.in3d.get_network(dem=...)` reads the walking network with `indoor_elements=True` and returns its nodes and edges. The edges have two new columns, `u_height` and `v_height`: the height in metres at the two ends of an edge, beside pyrosm's `u_level` and `v_level`. Their lines have a height at every vertex. Other arguments go to pyrosm's `get_network`; here `walking_areas=True` adds the lines across squares and halls.

The box holds the railway station, Forum and the Kamppi centre.

In [1]:
import cafein.sampledata.helsinki as helsinki
from pyrosm import OSM

# west, south, east, north
box = [24.9270, 60.1655, 24.9470, 60.1740]
osm = OSM(str(helsinki.osm_pbf), bounding_box=box)
nodes, edges = osm.in3d.get_network(dem=helsinki.dem, walking_areas=True)

columns = ["id", "vertical", "u_level", "v_level", "u_height", "v_height"]
edges[edges["vertical"] == "escalator"][columns].head()

,id,vertical,u_level,v_level,u_height,v_height
221,18378805,escalator,-2.0,-3.0,-7.665565,-13.664638
222,18378880,escalator,-1.0,0.0,-1.530364,4.035170
238,18379178,escalator,-1.0,0.0,-2.474509,3.345182
308,23648432,escalator,-1.0,0.0,-1.606828,4.018978
406,23653232,escalator,1.0,2.0,10.853458,13.640777


The first escalator runs from level -2 to level -3 of the metro station under the railway station, from -7.7 m to -13.7 m: 6 m for the level, as for every level below ground, since nothing else is known of that depth. Where both floors meet streets, the streets decide: the last escalator, in Forum, climbs 2.8 m from level 1 to level 2.

## Draw the network in metres

`osm.in3d.plot_network(dem=...)` draws the network at these heights. The figure is true to scale: a metre is as long upwards as sideways. The vertical axis shows metres, and the hover text of a point gives its height. `center` and `radius` keep the figure to what lies near a point, here 200 m around Forum.

The streets rise from the railway station in the north-east towards Kamppi in the south-west. The corridors of Forum (blue) lie where its doors meet the streets, and the passages of the metro station lie under them.

In [2]:
forum = (24.9385, 60.1690)
osm.in3d.plot_network(center=forum, radius=200, dem=helsinki.dem).show()

## Draw a route

A route found in a graph is drawn in metres in the same way: give `plot_network` the graph, the route and the terrain model. This route runs from the metro platform of the railway station to the door of Forum to Kukontori.

In [3]:
import networkx as nx

graph = osm.get_graph(
    "walking", graph_type="networkx", indoor_elements=True, walking_areas=True
)
platform = 5403677505  # on the metro platform of the railway station, level -3
door = 256257168  # a door of Forum to its courtyard Kukontori, level 2
route = nx.shortest_path(graph, platform, door, weight="length")

figure = osm.in3d.plot_network(
    graph=graph, route=route, center=forum, radius=200, dem=helsinki.dem
)
figure.show()

The route starts at -13.7 m on the platform and ends at 13.1 m at the door, which is the height of the street that ends there. Hover over the route to read its height along the way. A route is drawn in full, also beyond `radius`.

## How the heights are found

- A street follows the terrain model at every vertex. A street is an edge without a level that is not indoors, in a tunnel, on a bridge, below street level by its `layer`, a lift or a way tagged with several levels; a building passage counts as a street.
- An end of any other edge at a node of a street has the ground's height there.
- The other ends are placed from those, all together. An edge on one level keeps its two ends at one height. An edge between two levels, such as stairs, an escalator or a lift, wants the height between them: 6 m for each level below the ground floor and 4 m for each level above it. An edge without a level that is no street, such as a tunnel, a bridge or an indoor way, holds its ends together, so it runs straight between the streets at its ends. The heights that miss these wishes least are taken; a short edge counts more than a long one.
- What nothing joins to a street lies by its level from the ground at its place, 6 m for each level below the ground floor and 4 m for each level above it.

The heights are a drawing aid and have limits:

- OpenStreetMap holds no depths. A metro platform is placed 6 m per level under the floors that its stairs, escalators and lifts lead to, not at its real depth. Compared with the published depths of 41 underground platforms in 16 cities, platforms placed by their levels were about 5 m off in the median.
- A terrain model shows the ground without what is built on it. A street on a deck, or in a courtyard above a slope, follows the modelled ground under it, and a coarse model smooths a slope under a building: with the 10 m model of this page the doors on the two sides of Kukontori come out several metres apart.
- Where the terrain model has no height, an end's height is missing (NaN), the line of such an edge stays flat, and `plot_network` leaves the edge out.